# QLoRA Fine-Tuning
Instruction tuning for text->JSON transformation.

In [1]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import yaml
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

ROOT = Path('..').resolve()
CONFIG = yaml.safe_load((ROOT / 'config.yaml').read_text())

seed = CONFIG['seed']
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


C:\Users\Elena\anaconda3\envs\qlora-json\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_rows = read_jsonl(ROOT / CONFIG['dataset']['train_path'])

def prompt_format(row):
    return (
        '<|im_start|>system\nYou are a strict JSON generator. Return only JSON.\n<|im_end|>\n'
        f"<|im_start|>user\nInstruction: {row['instruction']}\nInput: {row['input']}\n<|im_end|>\n"
        f"<|im_start|>assistant\n{row['output']}<|im_end|>"
    )

train_texts = [prompt_format(r) for r in train_rows]
train_ds = Dataset.from_dict({'text': train_texts})
print('train samples:', len(train_ds))


train samples: 156


In [3]:
model_name = CONFIG['model_name']
trainer = None
tokenizer = None

if not torch.cuda.is_available():
    print(
        "QLoRA fine-tuning requires NVIDIA CUDA. "
        "CUDA is not available on this machine, so training setup is skipped."
    )
    print("Run this notebook on a CUDA-enabled machine to produce LoRA adapters.")
else:
    revision = CONFIG.get('base_model_revision', 'main')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name, revision=revision, trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        revision=revision,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )

    peft_config = LoraConfig(
        r=CONFIG['lora']['r'],
        lora_alpha=CONFIG['lora']['lora_alpha'],
        lora_dropout=CONFIG['lora']['lora_dropout'],
        target_modules=CONFIG['lora']['target_modules'],
        bias='none',
        task_type='CAUSAL_LM',
    )

    args = SFTConfig(
        output_dir=CONFIG['training']['output_dir'],
        learning_rate=float(CONFIG['training']['learning_rate']),
        num_train_epochs=CONFIG['training']['num_train_epochs'],
        per_device_train_batch_size=CONFIG['training']['batch_size'],
        gradient_accumulation_steps=CONFIG['training']['gradient_accumulation_steps'],
        warmup_steps=CONFIG['training']['warmup_steps'],
        logging_steps=CONFIG['training']['logging_steps'],
        max_seq_length=CONFIG['training']['max_seq_length'],
        save_strategy=CONFIG['training']['save_strategy'],
        report_to='none',
        seed=seed,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        peft_config=peft_config,
        args=args,
        dataset_text_field='text',
    )


C:\Users\Elena\anaconda3\envs\qlora-json\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Elena\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multi

RuntimeError: CUDA is required but not available for bitsandbytes. Please consider installing the multi-platform enabled version of bitsandbytes, which is currently a work in progress. Please check currently supported platforms and installation instructions at https://huggingface.co/docs/bitsandbytes/main/en/installation#multi-backend

In [4]:
if trainer is None:
    print('Training was skipped: CUDA is not available in this environment.')
else:
    start = time.time()
    train_result = trainer.train()
    elapsed_min = (time.time() - start) / 60

    print(f'Training completed in {elapsed_min:.2f} minutes')
    print('Final training loss:', train_result.training_loss)

    trainer.save_model(CONFIG['training']['output_dir'])
    tokenizer.save_pretrained(CONFIG['training']['output_dir'])
    print('Adapters saved to:', CONFIG['training']['output_dir'])


NameError: name 'trainer' is not defined